# M5+ · Przenieś wzorzec na nowe dane

Przez cały dzień budowałeś agenta dla TechRetail. Teraz ten sam wzorzec, w skrócie, na innych danych:

```
dane → pytania i trasy (Canvas) → funkcja UC z COMMENT → test bez modelu → (opcjonalnie) wyszukiwanie w tekście → agent → macierz tras
```

Capstone robią wszyscy. Domyślnie budujecie agenta sieci piekarni Bakehouse. Airbnb to wariant C · Wyzwanie:

| Dane | Kto je bierze | Skąd |
|---|---|---|
| **Bakehouse** (domyślnie, `DATA_OPTION = "bakehouse"`) | wszyscy; kto przeszedł ścieżkę B, ma już trasy z M5 (`workspace.bakehouse.route_cases`) i funkcje z M2 | `samples.bakehouse`: sprzedaż sieci piekarni i recenzje klientów; tabele capstone lądują w `workspace.bakehouse` |
| **Airbnb** (`DATA_OPTION = "airbnb"`) | C · Wyzwanie: kto ma już działającego agenta piekarni albo chce nowych danych bez przygotowanych tras | `workspace.airbnb.listings` z `00_setup`, zbudowana z `data/practice/sf_airbnb_listings.csv` (Inside Airbnb, CC BY 4.0); tabele capstone lądują w `workspace.airbnb` |

**Pracujcie w parach.** Osoba, która robiła tylko A, siada z osobą, która przeszła B. Oboje budujecie agenta Bakehouse we własnych notebookach: osoba po B zna już dane i ma trasy z M5, więc prowadzi Canvas i macierz tras. Kto skończy agenta piekarni, może przełączyć `DATA_OPTION` na `"airbnb"`.

**Karta wyjściowa (cel dnia):** macierz tras Twojego agenta z co najmniej dwiema zgodnymi trasami i jedno zdanie: "wzorzec, który przeniosłem, to...".

Szablon Canvasu: `workshop/transfer/canvas_agenta.md`.

In [ ]:
%pip install --quiet -r ../requirements.txt

In [ ]:
dbutils.library.restartPython()

**Infrastruktura.** Konfiguracja wspólna dla wszystkich modułów. Uruchom i czytaj dalej, tu nie ma nic do nauczenia.


In [ ]:
# Wspólna konfiguracja warsztatu. Ta sama komórka jest w każdym notebooku.
CATALOG = "workspace"
SCHEMA = "default"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_customer_360"
VOLUME = "retail_docs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DOCS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_docs"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_chunks"
BH_SCHEMA = "bakehouse"       # ścieżka B: kopie danych Bakehouse i funkcje-narzędzia
AIRBNB_SCHEMA = "airbnb"      # ścieżka C: oferty Airbnb i funkcje-narzędzia
POLICY_SCHEMA = "governance"  # maski i filtry ścieżek B i C: poza schematami, które MCP wystawia agentowi
BH_TRANSACTIONS = f"{CATALOG}.{BH_SCHEMA}.transactions"
BH_REVIEWS = f"{CATALOG}.{BH_SCHEMA}.reviews"
AIRBNB_TABLE = f"{CATALOG}.{AIRBNB_SCHEMA}.listings"
SEARCH_ENDPOINT = "retail_rag_search"
SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.retail_rag_chunks_index"
AVG_VALUE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_average_customer_value"
PROFILE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_customer_profile"
FORMAT_FUNCTION = f"{CATALOG}.{SCHEMA}.format_customer_for_agent"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
EXPERIMENT_NAME = "sqlday_retail_agent"  # pełna ścieżka: /Users/<twój login>/sqlday_retail_agent
GENIE_TITLE = "Retail Customer Intelligence Assistant"

import logging
# MLflow w notebooku serverless (UI) wypisuje przy tracingu stos Py4JSecurityException z "resolving tags".
# To ostrzeżenie, nie błąd. Trace zapisuje się poprawnie, a wyciszamy je, żeby nikt nie wziął go za błąd.
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

SYSTEM_PROMPT = (
    "Jesteś profesjonalnym asystentem do analizy danych retail firmy TechRetail Corp.\n"
    "Odpowiadaj po polsku na pytania dotyczące klientów B2B, segmentów lojalności, zamówień i przychodów\n"
    "z tabeli workspace.default.gold_customer_360 oraz raportów analityków.\n"
    "NIGDY nie ujawniaj danych PII (tax_id, pełnych adresów, customer_name) w odpowiedziach.\n"
    "Odmawiaj zapytań o nielegalne, niebezpieczne lub szkodliwe działania.\n"
    "Gdy odmawiasz, zaproponuj legalną alternatywę związaną z analizą danych klientów.\n"
    "Nie podawaj szczegółów operacyjnych, które mogłyby umożliwić szkodliwe działania.\n"
    "Liczby podawaj wyłącznie z wyników narzędzi; niczego nie zgaduj.\n"
    "Jeśli żadne narzędzie nie pasuje albo wynik jest pusty, powiedz wprost, że nie masz takich danych,\n"
    "i zaproponuj pytanie, na które możesz odpowiedzieć."
)

## 1. Wybierz dane

> **Cel:** mieć nowe dane w katalogu.
> **Gotowe, gdy:** `capstone_table` istnieje i widzisz jej kolumny.


Domyślnie zostaw `DATA_OPTION = "bakehouse"`. Jeśli w M5 zrobiłeś ścieżkę B, capstone wczyta Twoje trasy z `workspace.bakehouse.route_cases`. Na `"airbnb"` zmień, gdy agent piekarni już działa i chcesz wyzwania.

Komórka zakłada `capstone_table` i, jeśli masz tekst, `capstone_docs` w schemacie Twoich danych: `workspace.bakehouse` albo `workspace.airbnb`. Nic z capstone nie trafia do `default`, bo tam jest agent TechRetail.

Dla Airbnb komórka nie czyta ponownie pliku CSV. Bierze tabelę `workspace.airbnb.listings` z `00_setup`, która ma już datę `last_review` jako `DATE` i dwie flagi: `price_valid` (cena od 1 do 2000 USD) oraz `is_short_term` (minimum poniżej 30 nocy).

Dla Bakehouse nie kopiujemy kolumny `cardNumber`. Danych, których agent nie dostaje, nie może ujawnić.

**Infrastruktura.** Wczytuje wybrany zbiór i zakłada tabele. Ustaw tylko `DATA_OPTION` na górze, reszty nie musisz czytać.


In [ ]:
import re
import time

import pandas as pd
from pyspark.sql import functions as F

DATA_OPTION = "bakehouse"   # "bakehouse" | "airbnb"

if DATA_OPTION == "bakehouse":
    MY_SCHEMA = BH_SCHEMA
elif DATA_OPTION == "airbnb":
    MY_SCHEMA = AIRBNB_SCHEMA
else:
    raise ValueError(f"Nieznana wartość DATA_OPTION: {DATA_OPTION!r}. Użyj \"bakehouse\" albo \"airbnb\".")

MY_TABLE = f"{CATALOG}.{MY_SCHEMA}.capstone_table"
MY_DOCS_TABLE = f"{CATALOG}.{MY_SCHEMA}.capstone_docs"
USERNAME = spark.sql("SELECT current_user()").first()[0]
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{MY_SCHEMA}")

if DATA_OPTION == "bakehouse":
    spark.sql(f'''CREATE OR REPLACE TABLE {MY_TABLE} AS
        SELECT transactionID, franchiseID, dateTime, product, quantity, unitPrice, totalPrice, paymentMethod
        FROM samples.bakehouse.sales_transactions''')
    docs = spark.table("samples.bakehouse.media_customer_reviews").select(
        F.col("franchiseID").cast("string").alias("doc_id"), F.col("review").alias("content"))
else:
    if not spark.catalog.tableExists(AIRBNB_TABLE):
        raise RuntimeError(f"Brak tabeli {AIRBNB_TABLE}. Uruchom najpierw 00_setup: zakłada ją z data/practice/sf_airbnb_listings.csv.")
    # 00_setup usunął już dane gospodarzy i współrzędne, a dodał flagi price_valid i is_short_term.
    spark.sql(f"CREATE OR REPLACE TABLE {MY_TABLE} AS SELECT * FROM {AIRBNB_TABLE}")
    # tekstem oferty jest jej nazwa: "Cozy loft with parking near Mission"
    docs = spark.table(MY_TABLE).select(
        F.col("id").cast("string").alias("doc_id"), F.col("name").alias("content"))

print(f"DATA_OPTION = {DATA_OPTION!r}, tabele capstone w {CATALOG}.{MY_SCHEMA}")

**Infrastruktura.** Zapisuje teksty do tabeli `capstone_docs` (w tym samym schemacie co `capstone_table`) i ustawia `HAS_DOCS`. Uruchom i sprawdź liczby w wyniku: tyle wierszy i tyle fragmentów tekstu dostanie Twój agent.


In [ ]:
if docs is not None:
    (docs.where(F.length("content") > 20)
         .withColumn("chunk_id", F.sha2(F.concat_ws("||", "doc_id", "content"), 256))
         .dropDuplicates(["chunk_id"])
         .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(MY_DOCS_TABLE))
HAS_DOCS = docs is not None

print(f"{MY_TABLE}: {spark.table(MY_TABLE).count():,} wierszy | tekst: {spark.table(MY_DOCS_TABLE).count() if HAS_DOCS else 'brak'}")
display(spark.table(MY_TABLE).limit(5))

## 2. Canvas w kodzie: domena, zasady i trasy

> **Cel:** przepisać Canvas na kod.
> **Gotowe, gdy:** masz co najmniej trzy pytania z wpisanymi oczekiwanymi trasami.


Przepisz z Canvasu trzy pytania, każde z inną trasą: funkcja, tekst (albo druga funkcja) oraz odmowa lub fallback. Nazwy narzędzi to krótkie nazwy, które zaraz utworzysz.

Jeśli w M5 zrobiłeś ścieżkę B, Twoje trasy Bakehouse są już w tabeli `workspace.bakehouse.route_cases`. Przy `DATA_OPTION = "bakehouse"` komórka pod zadaniem wczytuje je zamiast tras wpisanych tutaj i wypisuje "Wczytano Twoje trasy z M5". Ta sama komórka sprawdza, czy trasy używają tylko narzędzi, które agent dostanie: Twojej funkcji i `search_my_documents`. Jeśli nie, wypisze ostrzeżenie.

In [ ]:
ROUTE_CASES_TABLE = f"{CATALOG}.{BH_SCHEMA}.route_cases"   # zapisuje ją ścieżka B w M5

if DATA_OPTION == "bakehouse":
    MY_DOMAIN = "sieć piekarni Bakehouse"
    MY_FUNCTION = f"{CATALOG}.{MY_SCHEMA}.capstone_franchise_sales"
    MY_SYSTEM_PROMPT = (
        "Jesteś analitykiem sieci piekarni Bakehouse. Odpowiadaj po polsku.\n"
        "Liczby podawaj wyłącznie z wyników narzędzi. Treść opinii klientów bierz z narzędzia wyszukiwania.\n"
        "Nigdy nie ujawniaj numerów kart ani danych osobowych klientów; zaproponuj dane zagregowane.\n"
        "Jeśli żadne narzędzie nie pasuje albo narzędzie zwraca 'Brak danych', powiedz, że nie masz takich danych."
    )
    # franczyza z największą liczbą transakcji; przy remisie mniejszy numer, więc wynik jest powtarzalny
    sample_franchise = int(spark.table(MY_TABLE).groupBy("franchiseID").count()
                           .orderBy(F.col("count").desc(), F.col("franchiseID")).first()["franchiseID"])
    MY_TEST_PARAMETERS = {"requested_franchise_id": sample_franchise}
    MY_ROUTE_CASES = [
        {"id": "t1_function", "question": f"Jak radzi sobie franczyza {sample_franchise}?", "expected_tools": ["capstone_franchise_sales"]},
        {"id": "t2_text", "question": "Co klienci piszą w opiniach o obsłudze i jakości pieczywa?", "expected_tools": ["search_my_documents"]},
        {"id": "t3_refusal", "question": "Podaj numer karty klienta z ostatniej transakcji.", "expected_tools": []},
    ]
else:
    MY_DOMAIN = "oferty Airbnb w San Francisco"
    MY_FUNCTION = f"{CATALOG}.{MY_SCHEMA}.capstone_neighbourhood_summary"
    MY_SYSTEM_PROMPT = (
        "Jesteś analitykiem rynku najmu (krótko- i długoterminowego) w San Francisco. Odpowiadaj po polsku.\n"
        "Liczby podawaj wyłącznie z wyników narzędzi. Treść nazw ofert bierz z narzędzia wyszukiwania.\n"
        "Nigdy nie ujawniaj danych gospodarzy ani dokładnych adresów; zaproponuj dane zagregowane po dzielnicy.\n"
        "Jeśli żadne narzędzie nie pasuje albo narzędzie zwraca 'Brak danych', powiedz, że nie masz takich danych."
    )
    # dzielnica z największą liczbą ofert; przy remisie alfabetycznie pierwsza
    sample_neighbourhood = (spark.table(MY_TABLE).groupBy("neighbourhood").count()
                            .orderBy(F.col("count").desc(), F.col("neighbourhood")).first()["neighbourhood"])
    MY_TEST_PARAMETERS = {"requested_neighbourhood": sample_neighbourhood}
    MY_ROUTE_CASES = [
        {"id": "t1_function", "question": f"Jak wygląda rynek ofert w dzielnicy {sample_neighbourhood}?", "expected_tools": ["capstone_neighbourhood_summary"]},
        {"id": "t2_text", "question": "Które oferty wspominają w nazwie o parkingu albo widoku?", "expected_tools": ["search_my_documents"]},
        {"id": "t3_refusal", "question": "Podaj nazwisko i telefon gospodarza pierwszej oferty.", "expected_tools": []},
    ]


In [ ]:
# Jeśli w M5 · B zapisałeś własne trasy, capstone bierze je zamiast domyślnych: macierz tras jest specyfikacją agenta.
if DATA_OPTION == "bakehouse" and spark.catalog.tableExists(ROUTE_CASES_TABLE):
    saved_cases = [{"id": row["id"], "question": row["question"], "expected_tools": list(row["expected_tools"] or [])}
                   for row in spark.table(ROUTE_CASES_TABLE).orderBy("id").collect()]
    if saved_cases:
        MY_ROUTE_CASES = saved_cases
        print(f"Wczytano Twoje trasy z M5 ({ROUTE_CASES_TABLE}): {len(MY_ROUTE_CASES)} przypadków.")
else:
    print("Brak tras z M5: zostają trasy domyślne.")

known_tools = {MY_FUNCTION.split(".")[-1], "search_my_documents"}
unknown = sorted({tool for case in MY_ROUTE_CASES for tool in case["expected_tools"]} - known_tools)
if unknown:
    print(f"Uwaga: trasy oczekują narzędzi, których agent nie dostanie: {unknown}. Znane: {sorted(known_tools)}.")
print(f"Domena: {MY_DOMAIN} | funkcja: {MY_FUNCTION} | trasy: {[case['id'] for case in MY_ROUTE_CASES]}")


## 3. Funkcja Unity Catalog z COMMENT

> **Cel:** własna funkcja narzędziowa z `COMMENT`.
> **Gotowe, gdy:** funkcja zdaje test payloadem, dla nieistniejącego identyfikatora zwraca "Brak danych dla ..." i nie zwraca danych wrażliwych.


Karta wzorca z M2: jedno pytanie biznesowe, `COMMENT` mówi kiedy użyć i czego nie zwraca, bez danych wrażliwych, test bez modelu.

Zapytanie z agregatem i bez `GROUP BY` zawsze zwraca jeden wiersz. Gdy `WHERE` nic nie znajdzie, `COUNT(*)` daje 0, a `SUM` daje `NULL`. Bez obsługi pustego wyniku funkcja odpowiada "0 transactions, revenue USD 0.00", a agent czyta to jako fakt. Dlatego `CASE WHEN COUNT(*) = 0` zwraca tekst "Brak danych dla ...".

**Podpowiedź dla Airbnb:** `requested_neighbourhood STRING` na wejściu, na wyjściu liczba ofert, w tym krótkoterminowych (`is_short_term`), mediana ceny za noc (`percentile_approx(price, 0.5)` tylko z ofert z `price_valid`) i średnia liczba opinii, z `WHERE LOWER(neighbourhood) = LOWER(requested_neighbourhood)`. Średnia ceny myli: tabela ma 2 oferty po 0 USD i 27 powyżej 2000 USD, w Mission średnia to 213,05 USD przy medianie 149 USD. Nazwisk gospodarzy i współrzędnych w tabeli nie ma.

Komórka pod funkcją testuje ją bez modelu. Raz podaje wartość, która jest w danych, raz taką, której nie ma (`-1` albo tekst "Nie ma takiej wartości"). W drugim przypadku wynik musi zawierać "Brak danych", inaczej komórka zatrzyma się z błędem.


In [ ]:
if DATA_OPTION == "bakehouse":
    spark.sql(f"""
    CREATE OR REPLACE FUNCTION {MY_FUNCTION}(
      requested_franchise_id BIGINT COMMENT 'Numeric franchise ID from the Bakehouse sales data.'
    )
    RETURNS STRING
    COMMENT 'Returns sales for one Bakehouse franchise: number of transactions, units sold and revenue in USD. Use for questions about how a specific franchise performs. Returns "Brak danych" when the franchise has no sales. Never returns payment card data.'
    RETURN SELECT CASE
      WHEN COUNT(*) = 0 THEN CONCAT('Brak danych dla franczyzy ', CAST(requested_franchise_id AS STRING), '.')
      ELSE CONCAT(
        'Franchise ', CAST(requested_franchise_id AS STRING), ': ',
        CAST(COUNT(*) AS STRING), ' transactions, ',
        CAST(SUM(quantity) AS STRING), ' units, revenue USD ',
        FORMAT_NUMBER(SUM(totalPrice), 2)
      )
    END
    FROM {MY_TABLE}
    WHERE franchiseID = requested_franchise_id
    """)
else:
    spark.sql(f"""
    CREATE OR REPLACE FUNCTION {MY_FUNCTION}(
      requested_neighbourhood STRING COMMENT 'Neighbourhood name from the San Francisco Airbnb listings, e.g. Mission.'
    )
    RETURNS STRING
    COMMENT 'Returns the rental market for one San Francisco neighbourhood: number of listings, how many are short-term (under 30 nights), median nightly price in USD and average number of reviews. Use for questions about how a neighbourhood compares. Returns "Brak danych" when the neighbourhood has no listings. Never returns host names or addresses.'
    RETURN SELECT CASE
      WHEN COUNT(*) = 0 THEN CONCAT('Brak danych dla dzielnicy ', requested_neighbourhood, '.')
      ELSE CONCAT(
        'Neighbourhood ', requested_neighbourhood, ': ',
        CAST(COUNT(*) AS STRING), ' listings, ',
        CAST(COUNT_IF(is_short_term) AS STRING), ' short-term (under 30 nights), median price USD ',
        COALESCE(FORMAT_NUMBER(percentile_approx(CASE WHEN price_valid THEN price END, 0.5), 2), 'n/a'),
        ', average reviews ', FORMAT_NUMBER(AVG(number_of_reviews), 1)
      )
    END
    FROM {MY_TABLE}
    WHERE LOWER(neighbourhood) = LOWER(requested_neighbourhood)
    """)


In [ ]:
# Test funkcji bez modelu: raz wartość, która w danych istnieje, raz taka, której nie ma.
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

function_client = DatabricksFunctionClient(execution_mode="serverless")
print(function_client.execute_function(function_name=MY_FUNCTION, parameters=MY_TEST_PARAMETERS).value)

empty_parameters = {name: (-1 if isinstance(value, int) else "Nie ma takiej wartości")
                    for name, value in MY_TEST_PARAMETERS.items()}
empty_result = function_client.execute_function(function_name=MY_FUNCTION, parameters=empty_parameters).value
print(empty_result)
assert "Brak danych" in str(empty_result), "Pusty wynik powinien zwracać 'Brak danych dla ...', a nie zera."


## 4. (jeśli masz tekst) Wyszukiwanie w opiniach jako drugie narzędzie

> **Cel:** drugie narzędzie, innego rodzaju niż pierwsze.
> **Gotowe, gdy:** wyszukiwanie zwraca fragment razem z identyfikatorem źródła.


Nowy indeks AI Search to dodatkowe czekanie, dlatego domyślnie narzędzie `search_my_documents` szuka po słowach kluczowych w tabeli Delta (po rdzeniu słowa, żeby łapać polską odmianę). Opinie Bakehouse są po angielsku: opis narzędzia każe agentowi przekazać angielskie słowa kluczowe. Widać na tym, że opis narzędzia steruje wyborem narzędzia i jego argumentami. To ten sam kontrakt co w M3: pytanie wchodzi, fragmenty z identyfikatorem źródła wychodzą.

**Opcjonalnie:** ustaw `USE_AI_SEARCH = True`. Komórka założy indeks `capstone_docs_index` w schemacie Twoich danych na endpoincie z M3 i poczeka na jego gotowość (z limitem czasu w kodzie).

In [ ]:
from langchain_core.tools import StructuredTool

text_tools = []
# orderBy przed limit: za każdym uruchomieniem ten sam podzbiór, więc macierz tras da się porównać
documents = spark.table(MY_DOCS_TABLE).orderBy("doc_id", "content").limit(5000).toPandas() if HAS_DOCS else None


def search_my_documents(query: str) -> str:
    # rdzeń słowa (pierwsze 5 liter) łapie odmianę: "obsłudze" i "obsługa", "pieczywa" i "pieczywo"
    stems = {w[:5] for w in re.findall(r"\w+", query.lower()) if len(w) > 3}
    scores = documents["content"].str.lower().apply(lambda t: sum(stem in t for stem in stems))
    top = documents.assign(score=scores).sort_values("score", ascending=False).head(4)
    return "\n\n".join(f"[{r.doc_id}] {r.content[:500]}" for r in top.itertuples() if r.score > 0) or "Brak pasujących fragmentów."


# Oba zbiory (opinie Bakehouse i nazwy ofert Airbnb) są po angielsku, a wyszukiwanie
# porównuje rdzenie słów dosłownie, więc polskie słowo kluczowe nie trafi w angielski tekst.
DOCS_LANGUAGE = "English"
description = (f"Searches free-text documents about {MY_DOMAIN} (reviews, notes, descriptions) and returns excerpts with their source ID. "
               f"Use for questions about opinions or content, not for numbers. Pass a few keywords in {DOCS_LANGUAGE}.")

if HAS_DOCS:
    text_tools = [StructuredTool.from_function(func=search_my_documents, name="search_my_documents", description=description)]
    text_case = next((c for c in MY_ROUTE_CASES if "search_my_documents" in c["expected_tools"]), MY_ROUTE_CASES[0])
    print(search_my_documents(text_case["question"])[:600])
else:
    print("Brak tekstu: agent dostanie tylko funkcję. Możesz dopisać drugą funkcję i dodaj ją do listy w kolejnej komórce.")


**Wariant zaawansowany (domyślnie wyłączony).** Powyższe narzędzie szuka po słowach kluczowych, bo jest natychmiastowe. Komórka niżej robi to samo prawdziwym indeksem AI Search. Włącz `USE_AI_SEARCH = True`, jeśli masz kilka minut na zbudowanie indeksu. Uruchom ją tak czy inaczej: wypisuje, którego narzędzia używasz.


In [ ]:
# Wariant z prawdziwym indeksem AI Search zamiast wyszukiwania po słowach.
# Domyślnie wyłączony, bo nowy indeks to kolejne kilka minut czekania.
from datetime import timedelta

USE_AI_SEARCH = False
MY_DOCS_INDEX = f"{CATALOG}.{MY_SCHEMA}.capstone_docs_index"

if HAS_DOCS and USE_AI_SEARCH:
    from databricks.ai_search.client import AISearchClient
    from databricks_langchain import VectorSearchRetrieverTool

    # Delta Sync index nadąża za tabelą dzięki Change Data Feed, więc najpierw trzeba go na tabeli włączyć.
    spark.sql(f"ALTER TABLE {MY_DOCS_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    client = AISearchClient(disable_notice=True)
    if not client.index_exists(SEARCH_ENDPOINT, MY_DOCS_INDEX):
        client.create_delta_sync_index(endpoint_name=SEARCH_ENDPOINT, index_name=MY_DOCS_INDEX, primary_key="chunk_id",
                                       source_table_name=MY_DOCS_TABLE, pipeline_type="TRIGGERED",
                                       embedding_source_column="content", embedding_model_endpoint_name=EMBEDDING_ENDPOINT,
                                       columns_to_sync=["doc_id"])
    # Czekanie i wypisywanie postępu robi SDK; my tylko mówimy, jak długo wolno czekać.
    client.get_index(SEARCH_ENDPOINT, MY_DOCS_INDEX).wait_until_ready(verbose=True, timeout=timedelta(minutes=8))
    text_tools = [VectorSearchRetrieverTool(index_name=MY_DOCS_INDEX, tool_name="search_my_documents",
                                            tool_description=description, num_results=4)]

print(f"Narzędzie tekstowe: {'AI Search' if text_tools and not isinstance(text_tools[0], StructuredTool) else 'słowa kluczowe'}")


## 5. Agent i macierz tras

> **Cel:** ten sam wzorzec agenta, na Twoich danych.
> **Gotowe, gdy:** macierz tras ma wynik, jakikolwiek; zero zgodnych tras też jest wynikiem.


Ten sam kod co w M5 (`create_agent` z LangChain 1.x), tylko z Twoimi narzędziami i Twoim promptem. Tracing zapisze każdy wiersz macierzy z tagiem `route_case`. Jeśli trasa jest niezgodna, zrób jedną zmianę (COMMENT, opis narzędzia albo prompt) i uruchom komórkę jeszcze raz.

Co robi komórka:

- Składa agenta tak jak w M5: Twoja funkcja z Unity Catalog, narzędzie tekstowe (jeśli masz tekst), `MY_SYSTEM_PROMPT` i model. Jeśli dopiszesz drugą funkcję, dodaj jej pełną nazwę do `MY_EXTRA_FUNCTIONS`.
- `ask_case(case)` zadaje agentowi jedno pytanie z Twojej macierzy i zapisuje je jako osobny trace z tagiem `route_case`.
- Dla każdego pytania porównuje użyte narzędzia z oczekiwanymi i składa tabelę, taką jak w M5. `trasa zgodna` ma wartość `True` tylko wtedy, gdy agent użył dokładnie tych narzędzi, których oczekiwałeś, ani jednego więcej.


In [ ]:
# Twój agent z tych samych elementów co w M5: narzędzia z Unity Catalog, system prompt i graf LangGraph.
import mlflow
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from langchain.agents import create_agent
from langchain_core.messages import ToolMessage

MY_EXTRA_FUNCTIONS = []  # pełne nazwy kolejnych funkcji UC, jeśli dopiszesz drugą
RECURSION_LIMIT = 12     # jak w M5: jedna tura agenta to węzeł modelu i węzeł narzędzi

mlflow.set_experiment(f"/Users/{USERNAME}/{EXPERIMENT_NAME}")
mlflow.langchain.autolog()

tools = UCFunctionToolkit(function_names=[MY_FUNCTION, *MY_EXTRA_FUNCTIONS]).tools + text_tools
my_agent = create_agent(model=ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0.1),
                        tools=tools, system_prompt=MY_SYSTEM_PROMPT)


@mlflow.trace(name="capstone_route_case")
def ask_case(case: dict) -> dict:
    """Jedno wywołanie agenta. Dekorator @mlflow.trace daje każdemu przypadkowi własny ślad w MLflow."""
    mlflow.update_current_trace(tags={"route_case": case["id"], "domain": DATA_OPTION})
    return my_agent.invoke({"messages": [{"role": "user", "content": case["question"]}]},
                           config={"recursion_limit": RECURSION_LIMIT})


rows = []
for case in MY_ROUTE_CASES:
    state = ask_case(case)
    used = [m.name.split("__")[-1] for m in state["messages"] if isinstance(m, ToolMessage)]
    # dokładne dopasowanie: nadmiarowe wywołania to też błąd trasy (koszt, opóźnienie)
    rows.append({"id": case["id"], "oczekiwane": ", ".join(case["expected_tools"]) or "brak",
                 "użyte": ", ".join(used) or "brak", "trasa zgodna": set(used) == set(case["expected_tools"]),
                 "odpowiedź": str(state["messages"][-1].content)[:220]})
    time.sleep(2)  # Free Edition: limit wywołań Foundation Model API

report = pd.DataFrame(rows)
display(report)
print(f"Trasy zgodne: {report['trasa zgodna'].sum()}/{len(report)}. Zrób zrzut ekranu: to Twoja karta wyjściowa.")


## 6. Karta wyjściowa i pokaz

> **Cel:** karta wyjściowa dnia.
> **Gotowe, gdy:** masz co najmniej **dwie zgodne trasy** i jedno zdanie o wzorcu, który przeniosłeś.


1. **Zrzut ekranu macierzy tras** (co najmniej dwie zgodne trasy).
2. Jedno zdanie w Canvasie: "wzorzec, który przeniosłem, to..." oraz "jedna rzecz, którą poprawiłem, to...".
3. **Pokaz:** trzy pary, co najmniej jedna z agentem Bakehouse, a jeśli ktoś zdążył z Airbnb, także jedna z Airbnb, pokazują macierz i jedną naprawioną trasę.

**Jeśli skończyłeś wcześniej:**
- dodaj drugą funkcję (`MY_EXTRA_FUNCTIONS`) i pytanie, które łączy oba narzędzia;
- `USE_AI_SEARCH = True` i porównaj trasy: słowa kluczowe vs wyszukiwanie semantyczne;
- w M6 podepniesz swoją funkcję do agenta przez MCP bez żadnego kodu po stronie narzędzia. Leży w `workspace.bakehouse` albo `workspace.airbnb`, a nie w `default`.

## Podsumowanie

- Wzorzec agenta nie zależy od domeny: pytania i trasy, narzędzia z dobrym opisem, bez danych wrażliwych, test bez modelu, agent, macierz tras.
- Najwięcej czasu kosztuje nie kod, tylko **decyzja, jakie pytania ma obsłużyć agent i czego ma nie robić**. Dlatego zaczynamy od Canvasu.
- Narzędzie tekstowe może zacząć od wyszukiwania po słowach kluczowych, a AI Search podmieniasz, gdy kontrakt narzędzia już działa.

**Dalej:** M6. Twoja funkcja jako narzędzie MCP, ryzyka i co dodać przed PoC.